# Notebook 12 — Circuit Overlap as the Real Predictor
**CS 590NN | Amogh | Apr 25**

## The hypothesis

Existing findings:
- Trade-off real (NB6, rho = -0.41)
- NOT caused by KL anchor (NB8 falsified)
- NOT a 1:1 conservation law (NB10 falsified)
- Edit magnitude weakly predicts vulnerability (NB13, rho = +0.32)

**Cleanest remaining hypothesis:** A and B share circuit components. Editing B touches MLPs/heads holding A. Displacement is parameter-collision, not contention.

## What this notebook does

1. **Auto-fetch** v3 results from GitHub for pair indices
2. Re-run ACDC on each unique sample
3. Compute Jaccard between circuit_A and circuit_B per pair
4. Test if overlap predicts A_displaced

## Compute

~25-30 min on A100 (~30 sec per ACDC × ~58 unique samples).

### 12.0 Install (run once, restarts)

In [ ]:
import subprocess, sys, os
def pip(args): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + args)
pip(['numpy==1.26.4'])
pip(['transformer-lens', 'transformers', 'datasets', 'accelerate', 'einops', 'jaxtyping'])
pip(['matplotlib', 'scipy'])
print('Restarting...'); os.kill(os.getpid(), 9)

### 12.1 Imports + auto-fetch v3

In [ ]:
import torch, json, requests, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime
from dataclasses import dataclass
from scipy import stats
from transformer_lens import HookedTransformer

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
RESULTS_DIR = Path('results_nb12'); RESULTS_DIR.mkdir(exist_ok=True)
FIG_DIR = RESULTS_DIR / 'figures'; FIG_DIR.mkdir(exist_ok=True)
torch.manual_seed(42); random.seed(42); np.random.seed(42)
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    free, total = torch.cuda.mem_get_info()
    print(f'GPU: {torch.cuda.get_device_name(0)} ({free/1e9:.1f}/{total/1e9:.1f} GB free)')

# Fetch v3 from GitHub
REPO_RAW = 'https://raw.githubusercontent.com/rukmini-17/targeted-unlearning/main/circuit_pipeline/results'
V3_URL = f'{REPO_RAW}/sequential_editing_v3/sequential_edit_rows_v3_20260424_220951.json'

print(f'\nFetching v3 from GitHub...')
v3_data = requests.get(V3_URL, timeout=60).json()
v3_ok = [r for r in v3_data['rows'] if r.get('status') == 'ok']
v3_df = pd.DataFrame(v3_ok)
print(f'v3 OK rows: {len(v3_df)}')

pair_map = {}
for r in v3_ok:
    pid = r['pair_idx']
    if pid not in pair_map:
        pair_map[pid] = (r['A_idx'], r['B_idx'], r['condition'])
print(f'Unique pairs: {len(pair_map)}')

### 12.2 Load model

In [ ]:
MODEL_NAME = 'Qwen/Qwen3-0.6B'
model = HookedTransformer.from_pretrained(
    MODEL_NAME,
    center_unembed=True, center_writing_weights=False,
    fold_ln=True, refactor_factored_attn_matrices=False, device=DEVICE,
)
model.set_use_attn_result(True)
model.eval()
model.tokenizer.pad_token = model.tokenizer.eos_token
free, _ = torch.cuda.mem_get_info()
print(f'Loaded {MODEL_NAME}. GPU free: {free/1e9:.1f} GB')

### 12.3 Load CounterFact

In [ ]:
@dataclass
class EditSample:
    idx: int; prompt: str
    target_new: str; target_true: str

raw = requests.get('https://rome.baulab.info/data/dsets/counterfact.json', timeout=120).json()
def make_sample(idx):
    rr = raw[idx]['requested_rewrite']
    return EditSample(idx=idx, prompt=rr['prompt'].format(rr['subject']),
                      target_new=' '+rr['target_new']['str'], target_true=' '+rr['target_true']['str'])

all_indices = sorted(set([a for a, b, _ in pair_map.values()] + [b for a, b, _ in pair_map.values()]))
samples = {idx: make_sample(idx) for idx in all_indices}
print(f'Built {len(samples)} unique samples')

### 12.4 ACDC implementation

In [ ]:
class NativeACDC:
    def __init__(self, model, threshold=0.3, effect_floor=0.5):
        self.m, self.th, self.floor = model, threshold, effect_floor
    def _ld(self, logits, t, b):
        return (logits[0,-1,t] - logits[0,-1,b]).item()
    def run(self, prompt, target_true, target_new):
        m = self.m
        true_id = m.to_tokens(target_true, prepend_bos=False)[0,0].item()
        new_id  = m.to_tokens(target_new,  prepend_bos=False)[0,0].item()
        tokens  = m.to_tokens(prompt)
        corrupt = tokens.clone()
        if tokens.shape[1] > 2:
            corrupt[0, 1:-1] = torch.randint(1000, m.cfg.d_vocab-1, (tokens.shape[1]-2,))
        with torch.no_grad():
            cl, cc = m.run_with_cache(tokens)
            kl, kc = m.run_with_cache(corrupt)
        clean_score = self._ld(cl, true_id, new_id)
        eff = max(abs(clean_score - self._ld(kl, true_id, new_id)), self.floor)
        attn_heads, mlp_layers = [], []
        for L in range(m.cfg.n_layers):
            hn = f'blocks.{L}.attn.hook_result'
            if hn in cc:
                for h in range(m.cfg.n_heads):
                    def mk(h=h, n=hn):
                        ca = kc[n][:,:,h:h+1,:].clone()
                        def fn(v, hook): v[:,:,h:h+1,:] = ca; return v
                        return fn
                    with torch.no_grad():
                        pl = m.run_with_hooks(tokens, fwd_hooks=[(hn, mk())])
                    if abs(self._ld(pl, true_id, new_id) - clean_score) / eff >= self.th:
                        attn_heads.append((L, h))
            hn = f'blocks.{L}.hook_mlp_out'
            if hn in cc:
                kca = kc[hn].clone()
                def mk_mlp(kca=kca):
                    def fn(v, hook): return kca
                    return fn
                with torch.no_grad():
                    pl = m.run_with_hooks(tokens, fwd_hooks=[(hn, mk_mlp())])
                if abs(self._ld(pl, true_id, new_id) - clean_score) / eff >= self.th:
                    mlp_layers.append(L)
        del cc, kc, cl, kl; torch.cuda.empty_cache()
        return {'attn_heads': sorted(set(attn_heads)), 'mlp_layers': sorted(set(mlp_layers))}

acdc = NativeACDC(model, threshold=0.3, effect_floor=0.5)
print('ACDC ready')

### 12.5 Discover circuits

In [ ]:
started = datetime.now()
circuits = {}
for i, (idx, s) in enumerate(samples.items()):
    try:
        c = acdc.run(s.prompt, s.target_true, s.target_new)
        circuits[idx] = c
        print(f'[{i+1:>2}/{len(samples)}] idx={idx:>4}  n_attn={len(c["attn_heads"]):>3}  n_mlp={len(c["mlp_layers"]):>2}')
    except Exception as e:
        print(f'[{i+1:>2}/{len(samples)}] idx={idx:>4}  FAILED: {type(e).__name__}: {e}')
        circuits[idx] = None
elapsed = (datetime.now()-started).total_seconds()/60
print(f'Done in {elapsed:.1f} min')

with open(RESULTS_DIR / 'circuits.json', 'w') as f:
    json.dump({str(k): v for k, v in circuits.items()}, f, indent=2)

### 12.6 Compute overlap per pair

In [ ]:
def jaccard(a, b):
    if not a and not b: return 1.0
    if not a or not b: return 0.0
    return len(set(a) & set(b)) / len(set(a) | set(b))

overlap_rows = []
for pid, (a_idx, b_idx, cond) in pair_map.items():
    cA, cB = circuits.get(a_idx), circuits.get(b_idx)
    if cA is None or cB is None: continue
    attn_A = [tuple(x) for x in cA['attn_heads']]
    attn_B = [tuple(x) for x in cB['attn_heads']]
    mlp_A, mlp_B = cA['mlp_layers'], cB['mlp_layers']
    j_attn = jaccard(attn_A, attn_B)
    j_mlp = jaccard(mlp_A, mlp_B)
    n_shared_attn = len(set(attn_A) & set(attn_B))
    n_shared_mlp = len(set(mlp_A) & set(mlp_B))
    combined_A = attn_A + [('m', x) for x in mlp_A]
    combined_B = attn_B + [('m', x) for x in mlp_B]
    overlap_rows.append({
        'pair_idx': pid, 'A_idx': a_idx, 'B_idx': b_idx, 'condition': cond,
        'n_attn_A': len(attn_A), 'n_attn_B': len(attn_B),
        'n_mlp_A': len(mlp_A), 'n_mlp_B': len(mlp_B),
        'n_shared_attn': n_shared_attn, 'n_shared_mlp': n_shared_mlp,
        'jaccard_attn': j_attn, 'jaccard_mlp': j_mlp,
        'jaccard_combined': jaccard(combined_A, combined_B),
    })
ov = pd.DataFrame(overlap_rows)
print(f'Overlap for {len(ov)} pairs')
print(ov[['jaccard_attn','jaccard_mlp','jaccard_combined','n_shared_mlp']].describe().round(3))

### 12.7 Merge with v3 retention and test

In [ ]:
v3_df['A_displaced'] = (v3_df['p_new_A_postA'] - v3_df['p_new_A_postAB']).clip(lower=0)
merged = v3_df.merge(ov, on='pair_idx', suffixes=('_v3', ''))
print(f'Merged: {len(merged)} rows')

clean = merged[merged['p_new_A_postA'] > 0.5].copy()
print(f'Clean: {len(clean)}')

print('\n=== Spearman: circuit overlap vs A_displaced (POOLED) ===')
for col in ['jaccard_attn', 'jaccard_mlp', 'jaccard_combined', 'n_shared_attn', 'n_shared_mlp']:
    rho, p = stats.spearmanr(clean[col], clean['A_displaced'])
    sig = '***' if p<0.001 else ('**' if p<0.01 else ('*' if p<0.05 else ''))
    print(f'  {col:>20}: rho={rho:+.3f}  p={p:.4f}  {sig}')

print('\n=== Per-method ===')
for m in clean['method'].unique():
    sub = clean[clean['method']==m]
    if len(sub) < 5: continue
    rho, p = stats.spearmanr(sub['jaccard_combined'], sub['A_displaced'])
    sig = '***' if p<0.001 else ('**' if p<0.01 else ('*' if p<0.05 else ''))
    print(f'  {m:>11}  rho={rho:+.3f}  p={p:.4f}  n={len(sub)}  {sig}')

print('\n=== Subject overlap (condition) vs circuit overlap ===')
for c in clean['condition'].unique():
    sub = clean[clean['condition']==c]
    print(f'  {c:>10}  n={len(sub):>2}  jaccard_combined mean={sub["jaccard_combined"].mean():.3f}  median={sub["jaccard_combined"].median():.3f}')

### 12.8 Figures

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
color_map = {'ACDC': '#cc3333', 'ROMEtrace': '#3366cc', 'Random': '#33aa33'}
for m, color in color_map.items():
    sub = clean[clean['method']==m]
    if len(sub) == 0: continue
    ax.scatter(sub['jaccard_combined'], sub['A_displaced'], color=color, s=55, alpha=0.7,
               edgecolors='black', lw=0.4, label=f'{m} (n={len(sub)})')
rho, p = stats.spearmanr(clean['jaccard_combined'], clean['A_displaced'])
ax.set_xlabel('Jaccard(circuit_A, circuit_B) combined')
ax.set_ylabel('A_displaced')
ax.set_title(f'Circuit overlap vs A displacement\npooled rho={rho:+.3f}, p={p:.4f}')
ax.legend(); ax.grid(alpha=0.3)

ax = axes[1]
clean['overlap_bin'] = pd.cut(clean['jaccard_combined'], bins=[-0.01, 0.05, 0.15, 0.3, 1.01],
                              labels=['none', 'low', 'mid', 'high'])
groups = [clean[clean['overlap_bin']==b]['A_displaced'].values for b in clean['overlap_bin'].cat.categories]
bp = ax.boxplot(groups, labels=clean['overlap_bin'].cat.categories, patch_artist=True, widths=0.6)
for patch in bp['boxes']: patch.set_facecolor('#aaccee')
ax.set_xlabel('Circuit overlap (Jaccard combined, binned)')
ax.set_ylabel('A_displaced')
ax.set_title('A displacement by overlap bin')
ax.grid(alpha=0.3, axis='y')
fig.tight_layout(); fig.savefig(FIG_DIR / 'fig1_overlap_vs_displaced.png', dpi=140); plt.show()

### 12.9 Save and bundle

In [ ]:
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
rho, p = stats.spearmanr(clean['jaccard_combined'], clean['A_displaced'])
summary = {
    'notebook': 'Notebook 12 - Circuit Overlap Predictor',
    'timestamp': ts, 'model': MODEL_NAME,
    'data_source': 'pair indices auto-fetched from github.com/rukmini-17/targeted-unlearning',
    'n_pairs': len(ov), 'n_clean_rows': len(clean),
    'spearman_jaccard_combined_vs_A_displaced': {'rho': float(rho), 'p': float(p)},
    'mean_jaccard_combined': float(ov['jaccard_combined'].mean()),
    'mean_jaccard_attn': float(ov['jaccard_attn'].mean()),
    'mean_jaccard_mlp': float(ov['jaccard_mlp'].mean()),
    'verdict': ('overlap_predicts_displacement' if rho > 0.3 and p < 0.01
                else 'overlap_does_not_predict_displacement'),
}
with open(RESULTS_DIR / f'summary_nb12_{ts}.json', 'w') as f: json.dump(summary, f, indent=2)
ov.to_csv(RESULTS_DIR / f'overlap_per_pair_{ts}.csv', index=False)
clean.to_csv(RESULTS_DIR / f'merged_with_overlap_{ts}.csv', index=False)
print(json.dumps(summary, indent=2))

import shutil
bundle = f'nb12_results_{ts}'
shutil.make_archive(bundle, 'zip', RESULTS_DIR)
from google.colab import files
files.download(f'{bundle}.zip')